In [212]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# 1. Load and sort data
df = pd.read_csv("data.csv")
df["From"] = pd.to_datetime(df["From"])
df = df.sort_values("From").reset_index(drop=True)

# flu_col = ["H1", "H3", "B", "AandB"]
flu_col = ["H1", "H3", "AandB"]

for n in range(4):
    df[f"H1_Lag{n+1}"] = df["H1"].shift(n+1)
df["H1_volatility"] = df["H1"].shift(1).rolling(window=4).std()
df["H1_rolling_mean"] = df["H1"].shift(1).rolling(window=4).mean()

df = df.dropna().reset_index(drop=True)

X = df.drop(columns=["From"]+flu_col) 
y = df["H1"]

train_end = int(df.shape[0] * 0.8)
X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]

y_train_log = np.log1p(y_train)

grid = {
    "n_estimators": [x for x in range(100, 1000, 100)],
    "max_features": ["sqrt", "log2"],
    "max_depth": [3, 4, 5, 6, 7, 8, 9, 10],
    "random_state": [18, 24, 42],
    "min_samples_split": [2, 5], 
    "min_samples_leaf": [5, 10, 20, 30]
}

tscv = TimeSeriesSplit(n_splits=5)

rfr = RandomForestRegressor(random_state=42)
rfr_cv = RandomizedSearchCV(estimator=rfr, param_distributions=grid, cv=tscv, scoring="r2", n_jobs=-1)

rfr_cv.fit(X_train, y_train_diff)
print(f"Best Parameters: {rfr_cv.best_params_}")

model = rfr_cv.best_estimator_
importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

model.fit(X_train, y_train_log)

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"R2:  {r2_score(y_test, y_pred):.4f}")

ValueError: Found input variables with inconsistent numbers of samples: [998, 997]